<a href="https://colab.research.google.com/github/rajeshtikaddar/Internship/blob/main/Login_detection_System_Day_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Login Detection System**

Import Libraries

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

**LOad DataSet**

In [31]:
df = pd.read_csv("/content/cybersecurity_login_data_120_records.csv")

df.head(10)

,User,Failed_Logins,Country,Device_Type,Login_Time,Account_Age_Days,Suspicious_Activity
0,Sneha1,30.0,Russia,Laptop,01:47,45,Yes
1,Ayesha2,60.0,Russia,Laptop,00:13,33,Yes
2,Manish3,250.0,Russia,Mobile,05:44,176,Yes
3,Riya4,120.0,Unknown,Laptop,02:55,217,Yes
4,Neha5,5.0,USA,Desktop,10:48,420,No
5,Sneha6,NaN,Russia,Mobile,02:22,226,Yes
6,Vikash7,120.0,Unknown,Laptop,00:24,246,Yes
7,Ayesha8,5.0,Unknown,Mobile,19:02,271,No
8,Pooja9,1.0,Unknown,Desktop,21:24,303,No
9,Sourav10,2.0,Germany,Desktop,13:42,414,No


Data Preprocessing

In [32]:
#Handle missing values

df["Failed_Logins"] = df["Failed_Logins"].fillna(
    df["Failed_Logins"].median()
)

#Remove duplicate

df = df.drop_duplicates()

Features Engineering

In [33]:
#Create high risk country feature
high_risk =[
    "Russia",
    "China",
    "Unknown"
]

df["High_Risk_Country"] = df["Country"].apply(
    lambda x: 1 if x in high_risk else 0
)

#Create Night Login  Features

df["Hour"]= pd.to_datetime(df["Login_Time"]).dt.hour

df["Night_Login"] = df["Hour"].apply(
    lambda x: 1 if x < 6 else 0
)

#Create Login Risk Score

df["Login_Risk_Score"] = (
    df["Failed_Logins"]*0.5
    + df["High_Risk_Country"] *15
    + df["Night_Login"]*10
)

df.head(10)

/tmp/ipykernel_3581/1598653751.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"]= pd.to_datetime(df["Login_Time"]).dt.hour


,User,Failed_Logins,Country,Device_Type,Login_Time,Account_Age_Days,Suspicious_Activity,High_Risk_Country,Hour,Night_Login,Login_Risk_Score
0,Sneha1,30.0,Russia,Laptop,01:47,45,Yes,1,1,1,40.0
1,Ayesha2,60.0,Russia,Laptop,00:13,33,Yes,1,0,1,55.0
2,Manish3,250.0,Russia,Mobile,05:44,176,Yes,1,5,1,150.0
3,Riya4,120.0,Unknown,Laptop,02:55,217,Yes,1,2,1,85.0
4,Neha5,5.0,USA,Desktop,10:48,420,No,0,10,0,2.5
5,Sneha6,18.0,Russia,Mobile,02:22,226,Yes,1,2,1,34.0
6,Vikash7,120.0,Unknown,Laptop,00:24,246,Yes,1,0,1,85.0
7,Ayesha8,5.0,Unknown,Mobile,19:02,271,No,1,19,0,17.5
8,Pooja9,1.0,Unknown,Desktop,21:24,303,No,1,21,0,15.5
9,Sourav10,2.0,Germany,Desktop,13:42,414,No,0,13,0,1.0


Encoding Categorial Data

In [34]:
le = LabelEncoder()

df["Device_Type"] = le.fit_transform(
    df["Device_Type"]
    )
df["Suspicious_Activity"] = le.fit_transform(
    df["Suspicious_Activity"]
    )



Feature Selection

In [35]:
x = df[
[
    "Failed_Logins",
    "Device_Type",
    "Account_Age_Days",
    "High_Risk_Country",
    "Night_Login",
    "Login_Risk_Score"
]
]

y = df["Suspicious_Activity"]

**Train-Test Split**

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

Train the model

In [37]:
model = DecisionTreeClassifier(
    random_state=42
)

model.fit(
    X_train,
    y_train
  )

DecisionTreeClassifier(random_state=42)

Generate Predections

In [38]:
predictions = model.predict(
    X_test
)

**Evaluate Model**

In [39]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        predictions
    )
)

#Confusion Matrix

print(
   confusion_matrix(
       y_test,
       predictions
   )
)

#Classification Report

print(
   classification_report(
       y_test,
       predictions
   )
)

Accuracy: 1.0
[[ 9  0]
 [ 0 15]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00        15

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24



In [41]:
new_user = pd.DataFrame(
{
"Failed_Logins":[20],
"Device_Type":[1],
"Account_Age_Days":[40],
"High_Risk_Country":[1],
"Night_Login":[1],
"Login_Risk_Score":[35]
}
)

prediction = model.predict(
    new_user
)


print(prediction)

[1]
